# pipeline_diagram

> The Nextflow DAG as a drawn object. One declaration of nodes, their constituent
> processes and the routing between them; layouts are computed here at build time and
> emitted as static SVG, so the page never positions a node at runtime. Two densities
> ship in the report (overview for Methods, standard for Run diagnostics); the fuller
> expansion lives in `docs/diagrams/`.


In [ ]:
#| default_exp pipeline_diagram

In [ ]:
#| export
from __future__ import annotations

# Geometry is on a 4px grid: every coordinate, box and gap below is divisible by 4.
# Node boxes are fixed size on purpose -- the run overlay writes into pre-sized slots
# rather than resizing anything, which is what keeps the routing below verifiable.
NODE_W, NODE_H = 160, 64
COL_GAP, ROW_GAP = 72, 80
ROW_PITCH = NODE_H + ROW_GAP
TOP = 88

INK, MUTED, SOFT = "#1a2233", "#66708a", "#8a93a8"
PAPER, ACCENT = "#f7f8fa", "#4650dc"


def _x(col: int, x0: int = 40) -> int:
    """Left edge of a grid column."""
    return x0 + col * (NODE_W + COL_GAP)


def _y(row: int) -> int:
    """Top edge of a grid row."""
    return TOP + row * ROW_PITCH

In [ ]:
#| export
# Every node names the Nextflow processes it stands for. Where a node covers several
# (Ablate, Multimodal, Report) the report must not add their task counts together --
# summing three processes into "78/79" is arithmetically true and useless. The node
# reports how many processes it covers; the per-process truth is one click down.
# tests/test_pipeline_diagram.py asserts this mapping against the actual workflow,
# so a stage added to the DAG cannot leave the diagram quietly describing the old one.
PIPELINE_NODES: dict[str, dict] = {
    "label":      dict(name="Label", sub="KREVIEW_LABEL", procs=["KREVIEW_LABEL"]),
    "extract":    dict(name="Extract", sub="26 matrices", procs=["KREVIEW_EXTRACT"]),
    "select":     dict(name="Select", sub="GrootCV", procs=["KREVIEW_SELECT_SINGLE"]),
    "evalcpu":    dict(name="Evaluate · CPU", sub="rf · xgb", procs=["KREVIEW_EVAL_CPU_SINGLE"]),
    "evalgpu":    dict(name="Evaluate · GPU", sub="tabpfn · tabicl", procs=["KREVIEW_EVAL_GPU_SINGLE"]),
    "ablate":     dict(name="Ablate", sub="cpu + gpu · merge", style="optional",
                       procs=["KREVIEW_ABLATE_CPU_SINGLE", "KREVIEW_ABLATE_GPU_SINGLE",
                              "KREVIEW_MERGE_ABLATION"]),
    "fuse":       dict(name="Fuse", sub="parquet lake", style="store", procs=["KREVIEW_FUSE"]),
    "multimodal": dict(name="Multimodal", sub="prep · fit · merge",
                       procs=["KREVIEW_MULTIMODAL_PREP", "KREVIEW_MULTIMODAL_SINGLE_CPU",
                              "KREVIEW_MULTIMODAL_SINGLE_GPU", "KREVIEW_MULTIMODAL_ABLATION",
                              "KREVIEW_MULTIMODAL_MERGE"]),
    "report":     dict(name="Report", sub="scoreboard + HTML", style="focal",
                       procs=["KREVIEW_SCOREBOARD", "KREVIEW_REPORT"]),
    # overview-only aggregates
    "familywork": dict(name="Per-family work", sub="select · eval · ablate",
                       procs=["KREVIEW_SELECT_SINGLE", "KREVIEW_EVAL_CPU_SINGLE",
                              "KREVIEW_EVAL_GPU_SINGLE", "KREVIEW_ABLATE_CPU_SINGLE",
                              "KREVIEW_ABLATE_GPU_SINGLE", "KREVIEW_MERGE_ABLATION"]),
    "stack":      dict(name="Stack & report", sub="multimodal · report", style="focal",
                       procs=["KREVIEW_MULTIMODAL_PREP", "KREVIEW_MULTIMODAL_SINGLE_CPU",
                              "KREVIEW_MULTIMODAL_SINGLE_GPU", "KREVIEW_MULTIMODAL_ABLATION",
                              "KREVIEW_MULTIMODAL_MERGE", "KREVIEW_SCOREBOARD", "KREVIEW_REPORT"]),
}

# Sub-topology behind each compressed node, as ordered stages of parallel members.
CLUSTERS: dict[str, list[list[str]]] = {
    "ablate": [["KREVIEW_ABLATE_CPU_SINGLE", "KREVIEW_ABLATE_GPU_SINGLE"], ["KREVIEW_MERGE_ABLATION"]],
    "multimodal": [["KREVIEW_MULTIMODAL_PREP"],
                   ["KREVIEW_MULTIMODAL_SINGLE_CPU", "KREVIEW_MULTIMODAL_SINGLE_GPU"],
                   ["KREVIEW_MULTIMODAL_ABLATION"], ["KREVIEW_MULTIMODAL_MERGE"]],
    "report": [["KREVIEW_SCOREBOARD"], ["KREVIEW_REPORT"]],
    "familywork": [["KREVIEW_SELECT_SINGLE"],
                   ["KREVIEW_EVAL_CPU_SINGLE", "KREVIEW_EVAL_GPU_SINGLE"],
                   ["KREVIEW_MERGE_ABLATION"]],
    "stack": [["KREVIEW_MULTIMODAL_PREP"], ["KREVIEW_MULTIMODAL_MERGE"],
              ["KREVIEW_SCOREBOARD"], ["KREVIEW_REPORT"]],
}

_STYLES: dict[str | None, dict[str, str | None]] = {
    None:       dict(fill="#ffffff", stroke=INK, dash=None),
    "store":    dict(fill="rgba(26,34,51,0.05)", stroke=MUTED, dash=None),
    "optional": dict(fill="rgba(26,34,51,0.02)", stroke="rgba(26,34,51,0.20)", dash="4,3"),
    "focal":    dict(fill="rgba(70,80,220,0.08)", stroke=ACCENT, dash=None),
}

In [ ]:
#| export
def _path(d: str, dashed: bool = False, accent: bool = False) -> str:
    # nested same-type quotes inside an f-string need py3.12; this package targets 3.10
    stroke = "rgba(26,34,51,0.35)" if dashed else (ACCENT if accent else MUTED)
    width = "1" if dashed else "1.2"
    dash = ' stroke-dasharray="4,3"' if dashed else ""
    marker = "pd-arrow-accent" if accent else "pd-arrow"
    return (f'  <path d="{d}" fill="none" stroke="{stroke}" stroke-width="{width}"{dash}'
            f' marker-end="url(#{marker})"/>')


def _h(x1: int, y: int, x2: int, **kw) -> str:
    return _path(f"M {x1},{y} H {x2}", **kw)


def _v(x: int, y1: int, y2: int, **kw) -> str:
    return _path(f"M {x},{y1} V {y2}", **kw)


def _side_down(x1: int, y1: int, x2: int, y2: int, mx: int, **kw) -> str:
    """Right edge -> down -> left edge, quarter-arc bends (r=8)."""
    return _path(f"M {x1},{y1} H {mx - 8} Q {mx},{y1} {mx},{y1 + 8} "
                 f"V {y2 - 8} Q {mx},{y2} {mx + 8},{y2} H {x2}", **kw)


def _bottom_right(x1: int, y1: int, y2: int, x2: int, **kw) -> str:
    """Bottom edge -> down -> right into a left edge."""
    return _path(f"M {x1},{y1} V {y2 - 8} Q {x1},{y2} {x1 + 8},{y2} H {x2}", **kw)


def _right_up(x1: int, y1: int, bx: int, y2: int, **kw) -> str:
    """Right edge -> right -> up into a bottom edge (the gather move)."""
    return _path(f"M {x1},{y1} H {bx - 8} Q {bx},{y1} {bx},{y1 - 8} V {y2}", **kw)


def _label(mid_x: int, line_y: int, text: str, accent: bool = False) -> str:
    """Masked arrow label sitting 8px clear of its connector."""
    w = max(40, len(text) * 6 + 8)
    w += (4 - w % 4) % 4
    return (f'  <rect x="{mid_x - w // 2}" y="{line_y - 20}" width="{w}" height="12" rx="2" fill="{PAPER}"/>\n'
            f'  <text x="{mid_x}" y="{line_y - 11}" fill="{ACCENT if accent else SOFT}" font-size="8"'
            f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" text-anchor="middle"'
            f' letter-spacing="0.06em">{text}</text>')


def _node(key: str, col: int, row: int, x0: int = 40, chips: bool = True) -> str:
    n = PIPELINE_NODES[key]
    st = _STYLES[n.get("style")]
    x, y = _x(col, x0), _y(row)
    cx, cy = x + NODE_W // 2, y + NODE_H // 2
    dash = f' stroke-dasharray="{st["dash"]}"' if st["dash"] else ""
    sub_fill = ACCENT if n.get("style") == "focal" else MUTED
    # The status chip is a fixed 48x14 slot inside the box. It is empty in the markup:
    # the page fills it from the execution trace, and a slot that stays empty renders
    # as nothing rather than as success.
    slot = (f'\n    <rect class="pd-chip" x="{x + 104}" y="{y + 8}" width="48" height="14" rx="2"'
            f' fill="none" stroke="none"/>'
            f'\n    <text class="pd-chiptext" x="{x + 128}" y="{y + 18}" fill="{SOFT}" font-size="8"'
            f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" text-anchor="middle"></text>') if chips else ""
    return (f'  <g class="pd-node" data-node="{key}">'
            f'\n    <rect x="{x}" y="{y}" width="{NODE_W}" height="{NODE_H}" rx="6" fill="{PAPER}"/>'
            f'\n    <rect x="{x}" y="{y}" width="{NODE_W}" height="{NODE_H}" rx="6" fill="{st["fill"]}"'
            f' stroke="{st["stroke"]}" stroke-width="1"{dash}/>{slot}'
            f'\n    <text x="{cx}" y="{cy + 6}" fill="{INK}" font-size="12" font-weight="600"'
            f' font-family="\'Geist\', -apple-system, BlinkMacSystemFont, sans-serif" text-anchor="middle">{n["name"]}</text>'
            f'\n    <text x="{cx}" y="{cy + 22}" fill="{sub_fill}" font-size="9"'
            f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" text-anchor="middle">{n["sub"]}</text>'
            f'\n  </g>')

In [ ]:
#| export
_LEGEND = [("#ffffff", INK, None, "NEXTFLOW PROCESS"),
           ("rgba(26,34,51,0.02)", "rgba(26,34,51,0.20)", "4,3", "OPTIONAL STAGE"),
           ("rgba(26,34,51,0.05)", MUTED, None, "ARTIFACT STORE"),
           ("rgba(70,80,220,0.08)", ACCENT, None, "DELIVERABLE")]


def _legend(y: int, w: int) -> str:
    out = [f'  <line x1="40" y1="{y}" x2="{w - 40}" y2="{y}" stroke="rgba(26,34,51,0.10)" stroke-width="0.8"/>',
           f'  <text x="40" y="{y + 24}" fill="{MUTED}" font-size="8"'
           f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" letter-spacing="0.14em">LEGEND</text>']
    for i, (fill, stroke, dash, text) in enumerate(_LEGEND):
        x = 128 + i * 192
        d = f' stroke-dasharray="{dash}"' if dash else ""
        out += [f'  <rect x="{x}" y="{y + 16}" width="12" height="12" rx="2" fill="{fill}"'
                f' stroke="{stroke}" stroke-width="0.8"{d}/>',
                f'  <text x="{x + 20}" y="{y + 25}" fill="{MUTED}" font-size="8"'
                f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" letter-spacing="0.1em">{text}</text>']
    return "\n".join(out)


_DEFS = (f'  <defs>\n'
         f'    <marker id="pd-arrow" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto">'
         f'<polygon points="0 0, 8 3, 0 6" fill="{MUTED}"/></marker>\n'
         f'    <marker id="pd-arrow-accent" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto">'
         f'<polygon points="0 0, 8 3, 0 6" fill="{ACCENT}"/></marker>\n'
         f'  </defs>')


def _level_overview(chips: bool) -> dict:
    """Five nodes: what the pipeline does, with the fan-out named but not drawn."""
    x0 = 156
    y = _y(0) + NODE_H // 2
    def rx(c): return _x(c, x0) + NODE_W
    def lx(c): return _x(c, x0)
    parts = [_h(rx(0), y, lx(1)), _label((rx(0) + lx(1)) // 2, y, "LABELS"),
             _h(rx(1), y, lx(2), accent=True), _label((rx(1) + lx(2)) // 2, y, "FAN ×26", accent=True),
             _h(rx(2), y, lx(3)), _label((rx(2) + lx(3)) // 2, y, "JOIN ×26"),
             _h(rx(3), y, lx(4))]
    nodes = [_node(k, c, 0, x0, chips) for k, c in
             [("label", 0), ("extract", 1), ("familywork", 2), ("fuse", 3), ("stack", 4)]]
    return dict(w=1400, h=300, legend_y=232, body="\n".join(parts + nodes))


def _level_standard(chips: bool) -> dict:
    """Nine nodes: the scatter across CPU, GPU and optional ablation, and the gather."""
    edges = [
        _h(200, 120, 272), _label(236, 120, "LABELS"),
        _h(432, 120, 504, accent=True), _label(468, 120, "FAN ×26", accent=True),
        _h(664, 120, 736),
        _side_down(664, 136, 736, 264, mx=700),
        _bottom_right(584, 152, 408, 736, dashed=True), _label(964, 408, "IF ENABLED"),
        _h(896, 120, 968), _label(932, 120, "JOIN ×26"),
        _right_up(896, 264, 1024, 152),
        _right_up(896, 408, 1064, 152, dashed=True),
        _h(1128, 120, 1200),
        _v(1280, 152, 232),
    ]
    nodes = [_node(k, c, r, 40, chips) for k, c, r in
             [("label", 0, 0), ("extract", 1, 0), ("select", 2, 0), ("evalcpu", 3, 0),
              ("evalgpu", 3, 1), ("ablate", 3, 2), ("fuse", 4, 0), ("multimodal", 5, 0),
              ("report", 5, 1)]]
    return dict(w=1400, h=520, legend_y=464, body="\n".join(edges + nodes))


LEVELS = {"overview": (_level_overview, "kreview at five nodes",
                       "Label, extract, the per-family fan-out, the gather, and the report."),
          "standard": (_level_standard, "The kreview DAG",
                       "Label and extract, then work scatters per feature family across selection, "
                       "CPU and GPU evaluation and optional ablation, gathers in fuse, and ends in the report.")}

In [ ]:
#| export
_MW, _MH, _MGAP, _MPITCH = 148, 40, 48, 56


def mini_svg(cluster: str) -> str:
    """Sub-topology of one compressed node, for the drill-down panel."""
    stages = CLUSTERS[cluster]
    maxr = max(len(s) for s in stages)
    inner = maxr * _MH + (maxr - 1) * (_MPITCH - _MH)
    width = len(stages) * _MW + (len(stages) - 1) * _MGAP + 32
    height = inner + 32

    def pos(si: int, mi: int, n: int) -> tuple[int, int]:
        span = n * _MH + (n - 1) * (_MPITCH - _MH)
        return 16 + si * (_MW + _MGAP), int(16 + (inner - span) / 2 + mi * _MPITCH)

    arrows, boxes = [], []
    for si, stage in enumerate(stages[:-1]):
        nxt = stages[si + 1]
        for mi in range(len(stage)):
            x1, y1 = pos(si, mi, len(stage))
            for ti in range(len(nxt)):
                x2, y2 = pos(si + 1, ti, len(nxt))
                # fan the attach points so a 1->2 or 2->1 step never stacks two strokes
                sy = int(y1 + _MH / 2 + (0 if len(nxt) == 1 else (ti - 0.5) * 16))
                dy = int(y2 + _MH / 2 + (0 if len(stage) == 1 else (mi - 0.5) * 16))
                if sy == dy:
                    arrows.append(_h(x1 + _MW, sy, x2))
                else:
                    mx = int(x1 + _MW + _MGAP / 2)
                    step = 8 if dy > sy else -8
                    arrows.append(_path(f"M {x1 + _MW},{sy} H {mx - 8} Q {mx},{sy} {mx},{sy + step} "
                                        f"V {dy - step} Q {mx},{dy} {mx + 8},{dy} H {x2}"))
    for si, stage in enumerate(stages):
        for mi, proc in enumerate(stage):
            x, y = pos(si, mi, len(stage))
            boxes.append(
                f'  <g class="pd-mnode" data-proc="{proc}">'
                f'\n    <rect x="{x}" y="{y}" width="{_MW}" height="{_MH}" rx="6" fill="{PAPER}"/>'
                f'\n    <rect x="{x}" y="{y}" width="{_MW}" height="{_MH}" rx="6" fill="#ffffff"'
                f' stroke="{INK}" stroke-width="1"/>'
                f'\n    <text x="{x + _MW // 2}" y="{y + 18}" fill="{INK}" font-size="9"'
                f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" text-anchor="middle">'
                f'{proc.replace("KREVIEW_", "")}</text>'
                f'\n    <text class="pd-mchip" x="{x + _MW // 2}" y="{y + 31}" fill="{SOFT}" font-size="8"'
                f' font-family="\'Geist Mono\', ui-monospace, SFMono-Regular, Menlo, monospace" text-anchor="middle"></text>'
                f'\n  </g>')
    return (f'<svg class="pd-mini" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg"'
            f' role="img" aria-labelledby="pd-mini-{cluster}-t pd-mini-{cluster}-d">\n'
            f'  <title id="pd-mini-{cluster}-t">{cluster} sub-pipeline</title>\n'
            f'  <desc id="pd-mini-{cluster}-d">The Nextflow processes behind the {cluster} node'
            f' and how they connect.</desc>\n{_DEFS}\n'
            + "\n".join(arrows + boxes) + "\n</svg>")

In [ ]:
#| export
def pipeline_svg(level: str, *, chips: bool = True) -> str:
    """Static SVG for one detail level.

    ``chips=False`` omits the status slots entirely — the Methods copy explains the
    pipeline and has no run to report, so it should not carry empty slots that look
    like missing data.
    """
    if level not in LEVELS:
        raise KeyError(f"unknown level {level!r}; have {sorted(LEVELS)}")
    build, title, desc = LEVELS[level]
    spec = build(chips)
    return (f'<svg class="pd-dag" data-level="{level}" viewBox="0 0 {spec["w"]} {spec["h"]}"'
            f' xmlns="http://www.w3.org/2000/svg" role="img"'
            f' aria-labelledby="pd-{level}-t pd-{level}-d">\n'
            f'  <title id="pd-{level}-t">{title}</title>\n'
            f'  <desc id="pd-{level}-d">{desc}</desc>\n{_DEFS}\n'
            f'  <rect width="100%" height="100%" fill="{PAPER}"/>\n'
            f'{spec["body"]}\n{_legend(spec["legend_y"], spec["w"])}\n</svg>')


def mini_store() -> str:
    """All sub-topologies in one hidden block; the page clones them into the panel."""
    blocks = "".join(f'<div data-cluster="{c}">{mini_svg(c)}</div>' for c in sorted(CLUSTERS))
    return f'<div id="pd-minis" hidden>{blocks}</div>'


def node_process_map() -> dict[str, list[str]]:
    """``{node key: [Nextflow process, ...]}`` — the page joins the trace on this."""
    return {k: list(v["procs"]) for k, v in PIPELINE_NODES.items()}


def node_names() -> dict[str, str]:
    return {k: v["name"] for k, v in PIPELINE_NODES.items()}


def declared_processes() -> set[str]:
    """Every Nextflow process the diagram claims to cover."""
    return {p for v in PIPELINE_NODES.values() for p in v["procs"]}

In [ ]:
#| hide
# smoke: both levels render, carry their nodes, and stay on the 4px grid
for _lvl in LEVELS:
    _svg = pipeline_svg(_lvl)
    assert _svg.startswith("<svg") and _svg.rstrip().endswith("</svg>")
    assert 'role="img"' in _svg and "<title" in _svg and "<desc" in _svg
assert "KREVIEW_MERGE_ABLATION" in declared_processes()
assert set(CLUSTERS) <= set(PIPELINE_NODES)
print(len(pipeline_svg("standard")), "chars")